# 01: GAO Risk Pattern Classification

**Goal:** Build a risk classifier from 5,000+ GAO audit recommendations to predict which agencies/topics have the highest risk of project failure.

**Data:** GAO Recommendations Database + GAO High Risk List

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
df = pd.read_csv('../data/gao_recommendations.csv')
print(f"Total recommendations: {len(df)}")
print(f"Open: {len(df[df['status'] == 'Open'])}")
print(f"Priority: {len(df[df['priority'] == 'Priority'])}")
df.head()

## Exploratory Analysis

In [ ]:
# Status distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['status'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Recommendation Status')
axes[0].set_ylabel('Count')

df['priority'].value_counts().plot(kind='bar', ax=axes[1], color=['#e67e22', '#3498db'])
axes[1].set_title('Priority Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Open rate by agency
agency_open = df.groupby('agency')['status'].apply(lambda x: (x == 'Open').mean()).sort_values(ascending=False)
agency_open.head(10).plot(kind='barh', color='#e74c3c')
plt.title('Open Recommendation Rate by Agency')
plt.xlabel('Open Rate')
plt.show()

## Feature Engineering

In [ ]:
# Encode categorical features
le_agency = LabelEncoder()
le_topic = LabelEncoder()

df['agency_encoded'] = le_agency.fit_transform(df['agency_code'])
df['topic_encoded'] = le_topic.fit_transform(df['topic'])
df['years_open'] = 2024 - df['year_issued']
df['is_open'] = (df['status'] == 'Open').astype(int)
df['is_priority'] = (df['priority'] == 'Priority').astype(int)

features = ['agency_encoded', 'topic_encoded', 'year_issued', 'years_open', 'is_priority']
X = df[features]
y = df['is_open']

print(f"Features: {features}")
print(f"Open rate: {y.mean():.1%}")

## Train Risk Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Closed', 'Open']))

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance - Risk Classifier')
plt.show()

## Save Model

In [ ]:
import joblib
joblib.dump(clf, '../models/risk_classifier.pkl')
print("Model saved to ../models/risk_classifier.pkl")